# 09 — Business Insights

Synthesize EDA, clustering, GA vs PSO, TabNet, and SHAP into **actionable retention plays**.

In [1]:
from pathlib import Path
import random

import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "environment.yml").exists()),
    cwd,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

for d in (DATA_INTERIM, DATA_PROCESSED, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Random seed  : {RANDOM_SEED}")

Project root : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction
Random seed  : 42


In [2]:
import json
import pandas as pd
import matplotlib.pyplot as plt

eda = pd.read_csv(DATA_INTERIM / "ecomm_validated.csv")
ga = json.loads((DATA_PROCESSED / "ga_feature_selection.json").read_text(encoding="utf-8"))
pso = json.loads((DATA_PROCESSED / "pso_feature_selection.json").read_text(encoding="utf-8"))
winner = json.loads((DATA_PROCESSED / "nia_feature_selection_winner.json").read_text(encoding="utf-8"))
tabnet_metrics = pd.read_json(REPORTS_DIR / "07_tabnet_metrics.json")
baseline = pd.read_json(REPORTS_DIR / "04_baseline_metrics.json")
clusters = pd.read_csv(DATA_INTERIM / "02_kmeans_profiles.csv") if (DATA_INTERIM / "02_kmeans_profiles.csv").exists() else None

print("NIA winner:", winner["winner"])
print(tabnet_metrics)
print(baseline)

NIA winner: GA
   accuracy  precision    recall        f1   roc_auc    pr_auc  \
0  0.991119   0.950000  1.000000  0.974359  0.999966  0.999836   
1  0.990231   0.954315  0.989474  0.971576  0.999775  0.998945   

                   model  n_features  train_time_sec  
0  TabNet_winning_subset          17       28.338660  
1    TabNet_all_features          33       38.506647  
                model  accuracy  precision    recall        f1   roc_auc  \
0  LogisticRegression  0.788632   0.435135  0.847368  0.575000  0.886460   
1        RandomForest  0.891652   0.634921  0.842105  0.723982  0.952547   

     pr_auc  train_time_sec  
0  0.672383        0.037815  
1  0.826305        0.244662  


## Executive summary

1. Churn is **imbalanced (~17%)** — optimise Recall / F1 / PR-AUC, not Accuracy.
2. **Complaints + low satisfaction + short tenure** are the sharpest risk signals from EDA and SHAP.
3. **GA and PSO** both reduce dimensionality; the winning subset feeds TabNet.
4. **TabNet** provides the portfolio neural predictor; compare subset vs all-features metrics above.
5. Clusters identify CRM segments; supervised models own individual scores.

In [3]:
# Retention play scorecard figure
plays = pd.DataFrame([
    {"play": "Complaint recovery SWAT", "lever": "Complain / UnhappyComplain", "owner": "CX", "expected_effect": "Cut FN on high-risk complainers"},
    {"play": "New-customer onboarding", "lever": "Tenure / TenureBin", "owner": "Growth", "expected_effect": "Reduce early-life churn"},
    {"play": "Dormancy win-back", "lever": "DaySinceLastOrder / IsDormant", "owner": "CRM", "expected_effect": "Reactivate silent buyers"},
    {"play": "Cashback / coupon targeting", "lever": "CashbackAmount / CouponUsed", "owner": "Loyalty", "expected_effect": "Lift engagement efficiently"},
    {"play": "Logistics experience", "lever": "WarehouseToHome", "owner": "Ops", "expected_effect": "Improve delivery for remote tiers"},
])
plays.to_csv(REPORTS_DIR / "09_retention_plays.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis("off")
ax.table(cellText=plays.values, colLabels=plays.columns, loc="center", cellLoc="left")
ax.set_title("Retention playbook", pad=20)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "09_retention_playbook.png", dpi=150, bbox_inches="tight")
plt.show()
plays

C:\Users\ishan\AppData\Local\Temp\ipykernel_26320\3636375914.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,play,lever,owner,expected_effect
0,Complaint recovery SWAT,Complain / UnhappyComplain,CX,Cut FN on high-risk complainers
1,New-customer onboarding,Tenure / TenureBin,Growth,Reduce early-life churn
2,Dormancy win-back,DaySinceLastOrder / IsDormant,CRM,Reactivate silent buyers
3,Cashback / coupon targeting,CashbackAmount / CouponUsed,Loyalty,Lift engagement efficiently
4,Logistics experience,WarehouseToHome,Ops,Improve delivery for remote tiers


In [4]:
# NIA comparison snapshot
nia = pd.DataFrame([
    {"method": "Full features", "n_features": ga["n_total_features"], "cv_f1_rf": ga["cv_f1_full"]},
    {"method": "GA", "n_features": ga["n_selected"], "cv_f1_rf": ga["cv_f1_selected"]},
    {"method": "PSO", "n_features": pso["n_selected"], "cv_f1_rf": pso["cv_f1_selected"]},
])
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(nia["method"], nia["cv_f1_rf"], color=["#8d99ae", "#2a9d8f", "#e9c46a"])
ax.set_ylabel("RF CV F1")
ax.set_title(f"NIA feature selection (winner={winner['winner']})")
for i, r in nia.iterrows():
    ax.text(i, r["cv_f1_rf"] + 0.005, f"n={int(r['n_features'])}", ha="center")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "09_nia_summary.png", dpi=150)
plt.show()
nia

C:\Users\ishan\AppData\Local\Temp\ipykernel_26320\53520769.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,method,n_features,cv_f1_rf
0,Full features,33,0.645282
1,GA,17,0.688650
2,PSO,18,0.676188


## How to use the scores

1. Score customers weekly with the saved TabNet model on the winning feature set.
2. Prioritise outreach by **predicted probability × expected LTV** (not probability alone).
3. Route `UnhappyComplain` / high-complaint cases to CX before marketing discounts.
4. Re-run GA/PSO only when feature distributions drift materially (new categories, new app versions).

## LinkedIn-ready takeaways

- Deep EDA + clustering before any metaheuristic.
- GA (evolutionary) vs PSO (swarm) for the same fitness — fair NIA comparison.
- TabNet as the attentive ANN classifier on the reduced feature set.
- SHAP turns black-box scores into retention language.

**Pipeline complete:** notebooks `01`–`09`.